# Module 10: Important Modules

**Utrains Python Fundamentals** &middot; lab notebook

Part 5 of the course: Working with the Outside World.

## What you will be able to do by the end

- Import a module three different ways and know when to use each
- Use time, os, datetime and math from the standard library
- Separate config from secrets, and keep API keys out of your code
- Recognise what openai, langchain and langgraph each do

## How to use this notebook

Run every cell in order with **Shift + Enter**. Read the markdown before each
block, then run the code and compare what you see against what you expected.

Two cells in this notebook are marked **Your turn**. They contain `____` where
a piece of the syntax is missing. They will fail if you run them as they are.
That is deliberate. Replace each `____`, then run the cell until it succeeds.

The last section is the **Lab**. It is a short task with no code written for
you, so you have to put the module together yourself.

A module is a file of pre-written Python code that you bring into your own
program with `import`. This is where most of Python's real power comes from:
you rarely have to write something from scratch.

## What import actually does

`import` loads a module so its functions and variables become available in your
file. Without importing it first, Python has no idea the name exists.

Three patterns:

- `import module_name` gives you `module_name.something`
- `import module_name as alias` lets you use a shorter name
- `from module_name import something` brings in just one name directly

In [ ]:
import datetime as dt
from math import sqrt

print(dt.date.today())
print(sqrt(16))

## Measuring time

In [ ]:
import time

t0 = time.time()
time.sleep(0.3)
print(f"Waited {time.time() - t0:.1f}s")

## Working with the operating system

In [ ]:
import os

print("current directory:", os.getcwd())
print("files here:", sorted(os.listdir("."))[:10])

---

### Your turn 1

Measure how long a fake health check takes, and print the answer to two decimal places. Two blanks: the import and the timing call.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
import time


def run_healthcheck():
    time.sleep(0.25)
    return "healthy"


start = time.time()
status = run_healthcheck()
elapsed = time.time() - start

print(f"health check returned {status} in {elapsed:.2f}s")

## Config values versus secrets

**Config** is safe to keep in your code, such as which model to use. **Secrets**
such as API keys should never be committed to source control. Keep them in a
`.env` file and load them with `dotenv`.

```python
from dotenv import load_dotenv
import os

load_dotenv()

MODEL = "claude-sonnet-4-6"                      # config, fine to commit
api_key = os.environ.get("ANTHROPIC_API_KEY")    # secret, never commit

print("API key loaded:", bool(api_key))          # never print the raw key
```

The `.env` file itself, kept out of version control by `.gitignore`:

```
ANTHROPIC_API_KEY=sk-...
```

The cell below does the same job using only the standard library, so it runs
here without installing anything. Notice it prints whether the key exists, and
never the key itself.

In [ ]:
import os

MODEL = "claude-sonnet-4-6"
api_key = os.environ.get("ANTHROPIC_API_KEY")

print("model:", MODEL)
print("API key loaded:", bool(api_key))

## Setting up a virtual environment

Everything so far has needed only the standard library, which ships with
Python. The packages below do not.

A virtual environment is a folder holding its own copy of installed packages,
separate from everything else on your machine. Without one, every project
shares the same packages, and installing one project's dependencies can quietly
break another's.

```bash
uv venv                        # creates a .venv folder here

source .venv/bin/activate      # Linux / macOS
.venv\Scripts\activate         # Windows

uv pip install openai langchain langgraph

deactivate                     # leave when you are done
```

> Create a fresh virtual environment per project. You can delete the `.venv`
> folder and start clean without touching anything else on your machine.

## API, SDK and client, defined

- **API** the remote interface your program talks to over the network
- **SDK** the official library you import so you do not write raw HTTP calls,
  such as `anthropic` or `openai`
- **client** the object the SDK gives you to make calls, such as
  `client = Anthropic()`

## The three packages you will meet

These need network access and an API key, so they are shown here as reference
rather than run. The cell after them simulates the same shapes with plain
Python, so you can see the structure without a key.

**openai** &mdash; the official SDK for OpenAI models:

```python
from openai import OpenAI

client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
)
print(response.choices[0].message.content)
```

**langchain** &mdash; wraps different providers behind one interface:

```python
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini")
response = llm.invoke([HumanMessage(content="Say hello in one sentence.")])
print(response.content)
```

**langgraph** &mdash; structures a program as a graph of steps called nodes,
connected by edges. Each node is a plain function that receives the current
state and returns an update to it:

```python
from langgraph.graph import StateGraph, END

def greet_node(state):
    return {"message": f"Hello, {state['name']}!"}

graph = StateGraph(dict)
graph.add_node("greet", greet_node)
graph.set_entry_point("greet")
graph.add_edge("greet", END)

app = graph.compile()
result = app.invoke({"name": "Alice"})
print(result["message"])
```

In [ ]:
# A stand-in for the graph above, using nothing but the standard library.
# The point is the shape: a node is a function, state flows through it.

def greet_node(state):
    return {"message": f"Hello, {state['name']}!"}


def run_graph(nodes, state):
    for node in nodes:
        state.update(node(state))
    return state


result = run_graph([greet_node], {"name": "Alice"})
print(result["message"])

## LangChain agents compared with LangGraph

Both can build an **agent**, an AI program that decides what to do rather than
always running the same fixed steps. They hand you different amounts of
control.

LangChain's agent tools give you a ready made loop: hand it a model and a list
of tools and it repeatedly asks the model what to do next until it decides it
is done. Faster to set up for a standard pattern.

LangGraph gives you the loop itself instead of hiding it. You define the nodes,
the edges and the state, which means you control exactly when to loop, branch,
pause for a human, or stop. More code to wire up, but every step is visible.

Complex or multi-agent workflows tend to move toward LangGraph as they grow,
since a single hidden loop stops being enough control.

> There is no wrong choice. Reach for LangChain's agent tools for a quick
> standard tool-calling loop. Reach for LangGraph when you need to see and
> control the steps yourself.

## A tiny agent

Underneath, every agent does the same three things: read the state, decide what
to do, and return an update. The `if` below stands in for a model deciding
which tool to call.

In [ ]:
def get_weather(city):
    # pretend this calls a real weather API
    return f"It is sunny in {city}."


def agent_node(state):
    question = state["question"]
    if "weather" in question.lower():
        answer = get_weather("Boston")
    else:
        answer = "I can only answer weather questions right now."
    return {"answer": answer}


print(agent_node({"question": "What is the weather like?"})["answer"])
print(agent_node({"question": "Who won the game?"})["answer"])

---

### Your turn 2

Extend the tiny agent with a second condition so it can also answer a simple addition question, and fall through to the refusal otherwise.

Replace each `____` below, then run the cell. It will not run until you do.

In [ ]:
def agent_node(state):
    question = state["question"].lower()

    if "weather" in question:
        answer = get_weather("Boston")
    elif "plus" in question:
        answer = "That is 4."
    else:
        answer = "I can only answer weather or simple maths right now."

    return {"answer": answer}


for q in ["What is the weather like?", "what is 2 plus 2", "Who won the game?"]:
    print(q, "->", agent_node({"question": q})["answer"])

---

## Lab: A timed, config-driven health reporter


Write a small script inside this notebook that does four things.

Read a model name from an environment variable called `LAB_MODEL`, falling back
to `"gpt-4o-mini"` when it is not set. Print the model, and separately print
whether an API key called `LAB_API_KEY` is present, without ever printing its
value.

Use `os.listdir` to count how many files sit in the current folder.

Time a fake `run_healthcheck()` function that sleeps for a fraction of a second
and returns a status, and print the elapsed time to two decimal places.

Stamp the report with today's date using `datetime`.

Print all of it as one tidy report block.


**Done when:**

- [ ] The model comes from the environment with a fallback
- [ ] The key is reported as present or absent, never printed
- [ ] os is used to count files in the folder
- [ ] time measures the health check, datetime stamps the report

Write your answer in the cell below. There is no starter code on purpose.

In [ ]:
# Your lab answer goes here.

---

## Practice exercises

Work through these on your own after the lab. They come straight from the
course reference guide, so the wording matches what you will see there.

1. Use the os module to list every file in the current deployment directory.
2. Use dotenv to load a cloud provider's API key from a .env file, printing only whether it loaded, never the key itself.
3. Use the time module to measure how long a fake health check function takes to run.
4. Extend the tiny agent above with a second condition, so it also answers a simple math question like "what is 2 plus 2".

---

*Utrains &middot; support@utrains.org &middot; https://utrains.org*